# 10 — SAR ships: SSDD and HRSID

Radar is a different sensor altogether. This lesson takes the optical DOTA detector to synthetic aperture radar (SAR): it explores the two standard SAR ship benchmarks, measures how badly the optical model does zero-shot, then fine-tunes it with the published 12-epoch recipes.

| | SSDD | HRSID |
|---|---|---|
| Chips | ~1,160, mixed sizes | 5,604, 800×800 |
| Ships | 2,456 | ~17,000 |
| Format | `dataset.format: ssdd` (VOC XML, COCO, or DOTA layouts) | `dataset.format: hrsid` (COCO JSON) |
| Split | official: file numbers ending in 1 or 9 are test | official train / test JSONs |
| Recipe | `rotated_faster_rcnn/ssdd_le90_1x.json`, keep-ratio 608 | `rotated_faster_rcnn/hrsid_le90_1x.json`, keep-ratio 800 |
| Published (held-out test mAP50) | **90.34%** | **78.55%** |
| Get it | [Official-SSDD](https://github.com/TianwenZhang/Official-SSDD) → `SSDD_DATA_ROOT` | [HRSID](https://github.com/chaozhong2010/HRSID) → `HRSID_DATA_ROOT` (`HRSID_JPG`) |

Either dataset is enough; the lesson uses whichever it finds. On Kaggle, attach your own copy (neither is mirrored as an official Kaggle dataset). Post: [v0.3.0: four datasets and ONNX](https://deeplearning.earth/posts/2026-09-21_oriented-det_v0_3_0_four_datasets_and_onnx/)

## What changes with radar

A SAR satellite sends microwave pulses and records what bounces back, so it sees through clouds and at night. The image is **backscatter intensity**, a single channel (replicated to three for an RGB backbone), with a grainy multiplicative noise called **speckle**. Ships are bright, because metal superstructures reflect strongly back to the sensor, and open sea is dark. Offshore scenes are therefore easy. **Inshore** scenes are hard: harbours, piers, and buildings are just as bright as ships.

The optical DOTA model has never seen any of this, yet it is a good starting point: its low-level filters and its idea of what an elongated ship looks like transfer. Both recipes fine-tune from `hf://rotated_faster_rcnn_dota_le90_1x` with `keep_ratio` resizing (lesson 08) — SAR chips are small already, so there is **no tiling**.

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd), flush=True)
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. Point at the data

In [ ]:
from oriented_det.data.hrsid import HRSIDDataset
from oriented_det.data.ssdd import SSDDDataset

RUN_SMOKE = False  # True: a short fine-tuning run on a GPU (a few minutes)

SETS = {
    "ssdd": dict(loader=SSDDDataset, root=kaggle_input("ssdd", "official-ssdd")
                 or Path(os.environ.get("SSDD_DATA_ROOT", "/path/to/data/Official-SSDD-OPEN")),
                 recipe="rotated_faster_rcnn/ssdd_le90_1x.json", published=90.34),
    "hrsid": dict(loader=HRSIDDataset, root=kaggle_input("hrsid")
                  or Path(os.environ.get("HRSID_DATA_ROOT", "/path/to/data/HRSID_JPG")),
                  recipe="rotated_faster_rcnn/hrsid_le90_1x.json", published=78.55),
}
data = {}
for name, s in SETS.items():
    if s["root"].is_dir():
        data[name] = {split: s["loader"](s["root"], split=split, difficult_strategy="keep") for split in ("train", "test")}
        print(f"{name}: train {len(data[name]['train'])}, test {len(data[name]['test'])} chips  ({s['root']})")
    else:
        print(f"{name}: not found at {s['root']}")
if not data:
    raise FileNotFoundError("Neither SSDD nor HRSID found: set SSDD_DATA_ROOT or HRSID_DATA_ROOT (see the table above).")
SAR = next(iter(data))  # the dataset used for sections 4-5; change to "hrsid" if you have both
print("using", SAR, "for the zero-shot test and fine-tuning")

## 2. Ships are small

Box length in pixels, per dataset. Compare with HRSC2016 (lesson 08), where the median ship spans a few hundred pixels.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, splits in data.items():
    lengths = [max(a.rbox.width, a.rbox.height) for s in splits["train"] for a in s.annotations]
    ax.hist(lengths, bins=np.linspace(0, 200, 41), alpha=0.6, label=f"{name} (median {np.median(lengths):.0f} px)")
    print(f"{name}: {len(lengths)} training ships, {len(lengths) / len(splits['train']):.1f} per chip")
ax.set_xlabel("ship length (px)"); ax.legend(); plt.show()

## 3. One chip each

In [ ]:
from PIL import Image
from oriented_det.utils.viz import DrawingSpec, draw_boxes

views, titles = [], []
for name, splits in data.items():
    s = max(list(splits["train"])[:50], key=lambda s: len(s.annotations))  # a busy chip
    img = Image.open(s.image_path).convert("RGB")
    views.append(draw_boxes(img, [a.rbox for a in s.annotations],
                            specs=[DrawingSpec(outline=(0, 255, 128), width=2)] * len(s.annotations)))
    titles.append(f"{name}: {s.image_path.name}, {len(s.annotations)} ships")
show(*views, titles=titles, size=6)

## 4. Zero-shot: the optical model on radar

Run the DOTA Hub checkpoint on test chips, keep only its `ship` class, and compute mAP50. This is the baseline fine-tuning has to beat.

In [ ]:
from oriented_det.data import Detection, GroundTruth, compute_oriented_map


def evaluate(model, config, class_names, samples, keep_class="ship"):
    """mAP50 of one class over dataset samples, keeping every detection down to score 0.05."""
    dets, gts = {}, {}
    for s in samples:
        gts[s.image_path.stem] = [GroundTruth(a.rbox, 0, keep_class, a.difficult, s.image_path.stem) for a in s.annotations]
        found = detect(model, config, class_names, Image.open(s.image_path).convert("RGB"), score_thr=0.05, classes=[keep_class])
        dets[s.image_path.stem] = [Detection(d["rbox"], float(d["score"]), 0, keep_class, s.image_path.stem) for d in found]
    return compute_oriented_map(dets, gts, class_names=[keep_class], show_progress=False)[0]

In [ ]:
N_TEST = 100  # test chips used for both evaluations below; None = all
subset = list(data[SAR]["test"])[:N_TEST]
dota_model, dota_config, dota_classes = hub_model("rotated_faster_rcnn_dota_le90_1x")
zero_shot = evaluate(dota_model, dota_config, dota_classes, subset)
print(f"zero-shot DOTA model on {len(subset)} {SAR} test chips: mAP50 {100 * zero_shot:.2f}% "
      f"(published after fine-tuning: {SETS[SAR]['published']}%)")

## 5. Fine-tune

The published recipe, patched for a smoke run: 32 chips, one epoch. The full recipe trains all chips for 12 epochs (learning-rate drops at epochs 8 and 11).

In [ ]:
from oriented_det.train.config import TrainingExperimentConfig

cfg = TrainingExperimentConfig.load(CONFIGS / SETS[SAR]["recipe"])
print("recipe:", SETS[SAR]["recipe"], "| init from", cfg.checkpoint.load_from_checkpoint,
      "|", cfg.preprocessing.resize_mode, cfg.preprocessing.target_size)
cfg.dataset.data_root = SETS[SAR]["root"]
cfg.dataset.max_train_samples = 32
cfg.dataset.max_val_samples = 16
cfg.training.num_epochs = 1
cfg.evaluation.compute_map_every_n_epochs = 1
smoke = WORK / f"10_{SAR}_smoke.json"
cfg.save(smoke)
print("wrote", smoke)

In [ ]:
def newest_run(model_type):
    """(checkpoint, config) of the most recent odet train run for this model type."""
    run_dir = max((WORK / "runs" / model_type).iterdir(), key=lambda p: p.stat().st_mtime)
    ckpt = max((run_dir / "checkpoints").glob("*.pth"), key=lambda p: p.stat().st_mtime)
    return ckpt, run_dir / "config.json"

In [ ]:
import gc

import torch
from oriented_det.runtime.checkpoint import load_model_from_checkpoint

if not RUN_SMOKE:
    print("RUN_SMOKE=False: skipping training. Set it to True in section 1 on a GPU.")
else:
    dota_model = None  # hand the GPU memory to the training subprocess
    gc.collect(); torch.cuda.empty_cache()
    run([*ODET, "train", "--config", smoke])
    ckpt, run_config = newest_run("rotated_faster_rcnn")
    tuned = load_model_from_checkpoint(str(ckpt), str(run_config), device=DEVICE)
    print(f"after the smoke fine-tune: mAP50 {100 * evaluate(*tuned, subset):.2f}% vs zero-shot {100 * zero_shot:.2f}%")

## 6. The full recipes

```bash
odet train --config oriented_det/configs/rotated_faster_rcnn/ssdd_le90_1x.json    # set dataset.data_root first
odet train --config oriented_det/configs/rotated_faster_rcnn/hrsid_le90_1x.json
make eval-val EXPERIMENT=runs/rotated_faster_rcnn/<timestamp>                       # held-out official test
```

Published held-out test mAP50 after the full 12 epochs: **90.34%** on SSDD (literature Faster R-CNN is often ~89%) and **78.55%** on HRSID. HRSID is harder: more inshore scenes, more small ships. Details: [SSDD](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/data.md#ssdd) and [HRSID](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/data.md#hrsid) in the data guide. There are no SAR checkpoints on the Hub yet.

## What to try next

- Split the SSDD test set into inshore and offshore chips (the official release lists them) and evaluate each: most of the error lives inshore.
- Train the Oriented R-CNN HRSID recipe (`oriented_rcnn/hrsid_le90_1x.json`) and compare with Faster R-CNN.
- `export_ssdd_to_dota` (in `oriented_det.data`) writes DOTA-format folders if you prefer the DOTA tooling.
- **Next:** [11 — ONNX export](11_onnx_export.ipynb).